# two_stage_reverse_hpo — Reverse Two-Stage (path B) + Optuna HPO

**목적**: 기존 path A (clf → reg, y>0 only) 와 다른 패러다임을 시도하여 stacking 다양성 확보.

**Path B 동작** (각 fold):
1. **Stage 1 회귀 (먼저)** — 모든 train die 사용
   - target = `log1p(y_die_broadcast)` (unit y → die broadcast)
   - **sample_weight**: `y_die==0 → w0` (Optuna 탐색), `y_die>0 → 1.0`  (★ weighted MSE)
   - LGBMRegressor(objective ∈ {regression, poisson, tweedie_1.2, tweedie_1.5})
   - 예측: `expm1 + clip≥0` → `reg_pred`
2. **Stage 2 분류 (나중)** — 모든 die 사용
   - target = `(y_die_broadcast > 0).astype(int)`
   - **입력 X 에 Stage 1 reg_pred 컬럼 추가** (D+1 features)
   - LGBMClassifier(objective='binary'), scale_pos_weight ∈ {1.0, 1.5, 2.43, 3.5} 탐색
3. **Final die pred** = `clf_proba × reg_pred`
4. **Unit pred** = `groupby(KEY).mean()`

**path A vs path B**:
- path A: clf 먼저, reg 는 y>0 die 만, final = P × reg
- path B: **reg 먼저** (모든 die + weighted MSE), clf 가 reg pred 보조 feature 사용

**Optuna 탐색 (14 axis, 50 trials)**:
- LGBM 11종 HP (회귀·분류 공유)
- `w0` (y=0 die weight) — log uniform [0.05, 1.0]
- `reg_objective` — categorical (4종)
- `clf_scale_pos_weight` — categorical (4종)
- target_transform = log1p 고정 (03b 일관성)

**고정 설정**: PP=log1p preset (03b 동일), KFold=5 unit-level shuffle SEED=42, CLIP_Y_EXTREME=True

**격리**: `4_output/_temp/two_stage_reverse/` 신규. 모듈 무수정.

**비교 기준**:
- 03b (path A die-level): val=0.005718, test=0.008417
- 03f (path A unit-agg): val=0.005742
- reg_only/lgbm: val=0.005731
- BagZIT plateau best (zit_only): val=0.005709
- Stacking 11-base: val=0.005701

**Note (leakage 인지)**: Stage 2 의 입력 보조 feature 는 같은 fold train 의 in-fold reg_pred (overfit 한 train predict). nested CV (5×5) 가 정석이지만 비용이 크므로 단순 stacking 표준 방식 채택. val/test 는 fold 모델의 깨끗한 inference.

## 1. 환경 + import

In [1]:
import os, sys, json

%run ../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import (
    PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR,
)
from utils.data import load_all, get_feat_cols, split_xs

MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from final.modules import preprocess

import lightgbm as lgb
import optuna
from sklearn.model_selection import KFold

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트


## 2. 설정 (path B + Optuna HPO)

PP는 03b의 log1p preset 그대로 (직접 비교 가능). HP/w0/reg_obj/clf_spw 는 Optuna 가 탐색.

In [2]:
EXP_ID   = 'ts-reverse-hpo-001'
EXP_MEMO = 'Reverse Two-Stage (reg → clf) + weighted MSE + Optuna HPO'
USER     = 'jh'

N_TRIALS = 300
N_FOLDS  = 5
CLIP_Y_EXTREME = True
TARGET_TRANSFORM = 'log1p'

OUT_DIR = os.path.join(OUTPUT_DIR, '_temp', 'two_stage_reverse')
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')
os.makedirs(OUT_DIR, exist_ok=True)

# ── 전처리 PARAMS (03b log1p preset 동일) ──
PARAMS = {
    'missing_threshold':          0.5,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.25,
    'spatial_max_dist':           5.0,
    'post_impute_corr_threshold': 0.99,
    'post_impute_corr_keep_by':   'std',
}

print(f'EXP_ID={EXP_ID}')
print(f'N_TRIALS={N_TRIALS} | N_FOLDS={N_FOLDS}')
print(f'TARGET_TRANSFORM={TARGET_TRANSFORM} | CLIP_Y_EXTREME={CLIP_Y_EXTREME}')
print(f'OUT_DIR={OUT_DIR}')
print(f'DB_PATH={DB_PATH}')
print(f'PARAMS keys: {list(PARAMS)}')

EXP_ID=ts-reverse-hpo-001
N_TRIALS=300 | N_FOLDS=5
TARGET_TRANSFORM=log1p | CLIP_Y_EXTREME=True
OUT_DIR=c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\two_stage_reverse
DB_PATH=c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\two_stage_reverse\optuna_jh_ts-reverse-hpo-001.db
PARAMS keys: ['missing_threshold', 'corr_threshold', 'corr_keep_by', 'add_indicator', 'indicator_threshold', 'spatial_max_dist', 'post_impute_corr_threshold', 'post_impute_corr_keep_by']


## 3. 데이터 로드 + Y clip

In [3]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개 샘플')

y_train_unit = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

print(f'\n[데이터 로드] xs={xs.shape}, X feat_cols={len(feat_cols)}')
print(f'  unit train={len(y_train_unit):,}, val={len(y_val_unit):,}, test={len(y_test_unit):,}')
print(f'  y_train: max={y_train_unit.max():.6f}, mean={y_train_unit.mean():.6f}, zero ratio={(y_train_unit==0).mean():.1%}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개 샘플

[데이터 로드] xs=(174572, 1091), X feat_cols=1087
  unit train=26,187, val=8,727, test=8,729
  y_train: max=0.097417, mean=0.002481, zero ratio=70.8%


## 4. 전처리 (03b log1p preset)

In [4]:
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PARAMS)
xs_train_die = pp['xs_train']
xs_val_die   = pp['xs_val']
xs_test_die  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

X_train_die = xs_train_die[feat_cols_clean].values.astype(np.float64)
X_val_die   = xs_val_die[feat_cols_clean].values.astype(np.float64)
X_test_die  = xs_test_die[feat_cols_clean].values.astype(np.float64)
uid_train_die = xs_train_die[KEY_COL].values
uid_val_die   = xs_val_die[KEY_COL].values
uid_test_die  = xs_test_die[KEY_COL].values
y_train_die_broadcast = pd.Series(uid_train_die).map(y_train_unit).values.astype(np.float64)
assert not pd.isna(y_train_die_broadcast).any(), 'unmapped train die y'
y_bin_die_broadcast = (y_train_die_broadcast > 0).astype(np.int32)

n_train_die = len(X_train_die)
n_val_die   = len(X_val_die)
n_test_die  = len(X_test_die)
print(f'\n[cleaning] feat_cols_clean={len(feat_cols_clean)}')
print(f'  X_train_die: {X_train_die.shape}, val: {X_val_die.shape}, test: {X_test_die.shape}')
print(f'  y_train_die_broadcast (unit y): mean={y_train_die_broadcast.mean():.6f}')
print(f'  y_bin_die (broadcasted y>0):    pos ratio={y_bin_die_broadcast.mean():.4f}')

# KFold split
unit_ids_train_unique = y_train_unit.index.values
kf_global = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf_global.split(unit_ids_train_unique))
print(f'\nfold split: {N_FOLDS} folds')

[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1033 (54개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1033
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 928개
    컬럼: 1033 → 928 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=50%
  제거: 5개, 잔여: 923개
    컬럼: 928 → 923 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 896개
    컬럼: 923 → 896 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 332개, 잔여: 564개
    컬럼: 896 → 564 (332개 제거)
    DataFrame: (104748, 622)

[결측 indicator] 4개 컬럼 추가 (결측률 >= 25%)
[공간 보간 imputation] 총 결측: 343,494
  train-only 모드: train 104,748 / 전체 174,572 행
  1단계 (공간 보간, dist<=5.0): 156,772개 채움 → 잔여: 186,722
  2단계 (lot 평균, train 기준): 105,526개 채움 → 잔여: 81,196
  3단계 (train 전체 평균): 81,196개 채움 → 잔여: 0

  [요약] 343,494 → 공간(156,772) → lot(105,526) → 전체(81,196) → 잔여(0)

[고상관 제거] threshold=0.99, keep_by=std (std)
  제거: 0개, 잔여: 564개
    [고상관 제거 2차 / imputation 후] threshold=0.99
    컬럼: 564 → 564 (0개 제거)
    DataFrame: (104748, 626)

클리닝 완

## 5. Path B 학습 함수 (1 fold용 helper)

Stage 1 reg (weighted MSE, log1p) → Stage 2 clf (X+reg_pred 보조 feature) → Final = clf × reg.

여러 X (val_split, val, test) 에 대해 한 번에 예측.

In [5]:
def _train_path_b(X_tr, y_tr_continuous, y_tr_bin, X_others, hp, w0, reg_obj, clf_spw):
    """Path B 1 fold 학습 + 예측.
    X_others: list of np.array (val_split, val, test 등)
    Returns: list of (prob, reg_y, final) per X in X_others.
    """
    # ── Stage 1: 회귀 (weighted MSE, log1p) ──
    sw = np.where(y_tr_continuous == 0, w0, 1.0)
    y_tr_log = np.log1p(y_tr_continuous)
    reg_params = dict(hp)
    if reg_obj.startswith('tweedie'):
        reg_params['objective'] = 'tweedie'
        reg_params['tweedie_variance_power'] = float(reg_obj.split('_')[1])
    else:
        reg_params['objective'] = reg_obj
    reg = lgb.LGBMRegressor(**reg_params)
    reg.fit(X_tr, y_tr_log, sample_weight=sw)

    # ── Stage 1 in-fold train predict (Stage 2 보조 feature) ──
    reg_train_log = reg.predict(X_tr)
    reg_train_y   = np.clip(np.expm1(reg_train_log), 0.0, None)
    X_tr_aug = np.hstack([X_tr, reg_train_y.reshape(-1, 1)])

    # ── Stage 2: 분류 (binary, X+reg_pred) ──
    clf_params = dict(hp)
    clf_params['objective'] = 'binary'
    clf_params['scale_pos_weight'] = clf_spw
    clf = lgb.LGBMClassifier(**clf_params)
    clf.fit(X_tr_aug, y_tr_bin)

    # ── 예측 ──
    results = []
    for X_o in X_others:
        reg_log_o = reg.predict(X_o)
        reg_y_o   = np.clip(np.expm1(reg_log_o), 0.0, None)
        X_o_aug   = np.hstack([X_o, reg_y_o.reshape(-1, 1)])
        prob_o = clf.predict_proba(X_o_aug)[:, 1]
        prob_o = np.clip(prob_o, 0.0, 1.0)
        final_o = prob_o * reg_y_o
        results.append((prob_o, reg_y_o, final_o))
    return results


def _mean_die_to_unit(pred_die, uid_die):
    """die-level → unit-level mean 집계."""
    unit_id = np.asarray(uid_die)
    unique_units, inverse = np.unique(unit_id, return_inverse=True)
    pred_sum = np.zeros(len(unique_units))
    cnt      = np.zeros(len(unique_units))
    np.add.at(pred_sum, inverse, pred_die)
    np.add.at(cnt,      inverse, 1.0)
    return pred_sum / cnt, unique_units


def _rmse_unit(pred_unit_arr, unique_units, y_unit_series):
    s = pd.Series(pred_unit_arr, index=unique_units).reindex(y_unit_series.index)
    return float(np.sqrt(np.mean((s.values - y_unit_series.values) ** 2)))

print('helpers 정의 완료')

helpers 정의 완료


## 6. Optuna objective + study 실행

objective: OOF unit RMSE.

각 trial 학습 = 5 fold × (Stage 1 + Stage 2) = 10 모델 학습. 03b 1 fold ≈ 60s 기준 → 1 trial ≈ 5~10분.

In [6]:
import time

def objective(trial):
    # ── LGBM HP (회귀·분류 공유) ──
    hp = dict(
        n_estimators=trial.suggest_int('n_estimators', 100, 3000),
        learning_rate=trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        num_leaves=trial.suggest_int('num_leaves', 8, 512),
        max_depth=trial.suggest_int('max_depth', 3, 14),
        min_child_samples=trial.suggest_int('min_child_samples', 5, 400),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        subsample_freq=1,
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.1, 1.0),
        reg_alpha=trial.suggest_float('reg_alpha', 1e-8, 30.0, log=True),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        min_split_gain=trial.suggest_float('min_split_gain', 1e-9, 1.0, log=True),
        path_smooth=trial.suggest_float('path_smooth', 0.0, 50.0),
        random_state=SEED,
        n_jobs=-1,
        verbose=-1,
    )
    # ── path B 전용 ──
    w0 = trial.suggest_float('w0', 0.05, 1.0, log=True)
    reg_obj = trial.suggest_categorical(
        'reg_objective',
        ['regression', 'poisson', 'tweedie_1.2', 'tweedie_1.5']
    )
    clf_spw = trial.suggest_categorical(
        'clf_scale_pos_weight', [1.0, 1.5, 2.43, 3.5]
    )

    oof_die_pred = np.full(n_train_die, np.nan)
    for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
        tr_units = unit_ids_train_unique[tr_uidx]
        vl_units = unit_ids_train_unique[vl_uidx]
        tr_die_mask = np.isin(uid_train_die, tr_units)
        vl_die_mask = np.isin(uid_train_die, vl_units)
        X_tr  = X_train_die[tr_die_mask]
        X_vl  = X_train_die[vl_die_mask]
        y_tr  = y_train_die_broadcast[tr_die_mask]
        yb_tr = y_bin_die_broadcast[tr_die_mask]

        results = _train_path_b(X_tr, y_tr, yb_tr, [X_vl], hp, w0, reg_obj, clf_spw)
        _, _, f_vl = results[0]
        oof_die_pred[vl_die_mask] = f_vl

    if np.isnan(oof_die_pred).any():
        raise RuntimeError('OOF die pred has NaN — fold coverage bug')

    oof_unit_arr, oof_unit_ids = _mean_die_to_unit(oof_die_pred, uid_train_die)
    train_rmse = _rmse_unit(oof_unit_arr, oof_unit_ids, y_train_unit)
    trial.set_user_attr('train_rmse', train_rmse)
    return train_rmse

study = optuna.create_study(
    direction='minimize',
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    load_if_exists=False,
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study.set_user_attr('exp_memo',  EXP_MEMO)
study.set_user_attr('path_type', 'B (reverse: reg → clf weighted MSE)')
study.set_user_attr('pp_source', '03b log1p preset')

print(f'=== Optuna HPO 시작 (N_TRIALS={N_TRIALS}) ===')
t0 = time.time()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print(f'\n[HPO 완료] {time.time()-t0:.0f}s')
print(f'best OOF RMSE = {study.best_value:.6f}')
print(f'best params:')
for k, v in study.best_trial.params.items():
    print(f'  {k:30s} = {v}')

=== Optuna HPO 시작 (N_TRIALS=300) ===


  0%|          | 0/300 [00:00<?, ?it/s]


[HPO 완료] 87773s
best OOF RMSE = 0.005495
best params:
  n_estimators                   = 277
  learning_rate                  = 0.011522668395128848
  num_leaves                     = 476
  max_depth                      = 12
  min_child_samples              = 22
  subsample                      = 0.9887429605486475
  colsample_bytree               = 0.7495115219470344
  reg_alpha                      = 2.3797498689851316e-07
  reg_lambda                     = 2.7319219718867617e-07
  min_split_gain                 = 0.07892551176975685
  path_smooth                    = 31.80451129069409
  w0                             = 0.17672657730420213
  reg_objective                  = regression
  clf_scale_pos_weight           = 2.43


## 7. Best params 로 5-fold 재학습 + die-level 캐쳐

In [7]:
best = study.best_trial.params
best_w0       = best['w0']
best_reg_obj  = best['reg_objective']
best_clf_spw  = best['clf_scale_pos_weight']
hp_best = {k: v for k, v in best.items() if k not in ['w0', 'reg_objective', 'clf_scale_pos_weight']}
hp_best.update(random_state=SEED, n_jobs=-1, verbose=-1, subsample_freq=1)

oof_die_prob   = np.full(n_train_die, np.nan)
oof_die_reg    = np.full(n_train_die, np.nan)
oof_die_pred   = np.full(n_train_die, np.nan)
val_die_prob   = np.zeros(n_val_die)
val_die_reg    = np.zeros(n_val_die)
val_die_pred   = np.zeros(n_val_die)
test_die_prob  = np.zeros(n_test_die)
test_die_reg   = np.zeros(n_test_die)
test_die_pred  = np.zeros(n_test_die)

print('=== Refit (best params) 5-fold ===')
t0 = time.time()
for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
    tr_units = unit_ids_train_unique[tr_uidx]
    vl_units = unit_ids_train_unique[vl_uidx]
    tr_die_mask = np.isin(uid_train_die, tr_units)
    vl_die_mask = np.isin(uid_train_die, vl_units)
    X_tr  = X_train_die[tr_die_mask]
    X_vl  = X_train_die[vl_die_mask]
    y_tr  = y_train_die_broadcast[tr_die_mask]
    yb_tr = y_bin_die_broadcast[tr_die_mask]

    results = _train_path_b(X_tr, y_tr, yb_tr,
                            [X_vl, X_val_die, X_test_die],
                            hp_best, best_w0, best_reg_obj, best_clf_spw)
    (p_vl, r_vl, f_vl), (p_v, r_v, f_v), (p_t, r_t, f_t) = results

    oof_die_prob[vl_die_mask] = p_vl
    oof_die_reg[vl_die_mask]  = r_vl
    oof_die_pred[vl_die_mask] = f_vl
    val_die_prob  += p_v / N_FOLDS
    val_die_reg   += r_v / N_FOLDS
    val_die_pred  += f_v / N_FOLDS
    test_die_prob += p_t / N_FOLDS
    test_die_reg  += r_t / N_FOLDS
    test_die_pred += f_t / N_FOLDS

    print(f'  fold {fold_idx+1}/{N_FOLDS} done ({time.time()-t0:.0f}s)')

assert not np.isnan(oof_die_prob).any()
assert not np.isnan(oof_die_reg).any()
assert not np.isnan(oof_die_pred).any()
print(f'\n[Refit 완료] {time.time()-t0:.0f}s')

=== Refit (best params) 5-fold ===
  fold 1/5 done (48s)
  fold 2/5 done (96s)
  fold 3/5 done (144s)
  fold 4/5 done (192s)
  fold 5/5 done (239s)

[Refit 완료] 239s


## 8. unit aggregate + RMSE

In [8]:
oof_unit_arr,  oof_unit_ids  = _mean_die_to_unit(oof_die_pred,  uid_train_die)
val_unit_arr,  val_unit_ids  = _mean_die_to_unit(val_die_pred,  uid_val_die)
test_unit_arr, test_unit_ids = _mean_die_to_unit(test_die_pred, uid_test_die)

oof_unit_s  = pd.Series(oof_unit_arr,  index=oof_unit_ids).reindex(y_train_unit.index)
val_unit_s  = pd.Series(val_unit_arr,  index=val_unit_ids).reindex(y_val_unit.index)
test_unit_s = pd.Series(test_unit_arr, index=test_unit_ids).reindex(y_test_unit.index)

oof_rmse  = float(np.sqrt(np.mean((oof_unit_s.values  - y_train_unit.values) ** 2)))
val_rmse  = float(np.sqrt(np.mean((val_unit_s.values  - y_val_unit.values)   ** 2)))
test_rmse = float(np.sqrt(np.mean((test_unit_s.values - y_test_unit.values)  ** 2)))

print('=' * 75)
print(f'  Reverse Two-Stage (path B) — best HPO 결과')
print('=' * 75)
print(f'  {"":12s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"unit RMSE":12s}  {oof_rmse:11.6f}  {val_rmse:11.6f}  {test_rmse:11.6f}')
print('-' * 75)
print(f'  비교 기준선 (path A 계열):')
print(f'    03b (die-level broadcast):        val=0.005718, test=0.008417')
print(f'    03f (unit-agg):                    val=0.005742')
print(f'    reg_only/lgbm (Stage 1 없음):     val=0.005731, test=0.008429')
print(f'    BagZIT plateau best (zit_only):   val=0.005709, test=0.008414')
print(f'    Stacking 11-base:                  val=0.005701, test=0.008408')
print('=' * 75)
if val_rmse < 0.0058:
    print('  → plateau 영역 안. stacking pool 추가 후보 (잔차 corr 검증 필요).')
elif val_rmse < 0.006:
    print('  → plateau 근처. residual 패턴이 다르면 stacking 도움 가능.')
else:
    print('  → plateau 밖. path B 패러다임이 path A 보다 불리.')

  Reverse Two-Stage (path B) — best HPO 결과
                        OOF          val         test
  unit RMSE        0.005495     0.005709     0.008412
---------------------------------------------------------------------------
  비교 기준선 (path A 계열):
    03b (die-level broadcast):        val=0.005718, test=0.008417
    03f (unit-agg):                    val=0.005742
    reg_only/lgbm (Stage 1 없음):     val=0.005731, test=0.008429
    BagZIT plateau best (zit_only):   val=0.005709, test=0.008414
    Stacking 11-base:                  val=0.005701, test=0.008408
  → plateau 영역 안. stacking pool 추가 후보 (잔차 corr 검증 필요).


## 9. 산출물 저장 (`_temp/two_stage_reverse/`)

In [9]:
def _build_die_df(uid_arr, die_id_arr, position_arr, prob, reg, pred, y_unit):
    df = pd.DataFrame({
        KEY_COL:     uid_arr,
        DIE_KEY_COL: die_id_arr,
        'position':  position_arr,
        'prob':      prob,
        'reg':       reg,
        'pred':      pred,
    })
    if y_unit is not None:
        df[TARGET_COL] = df[KEY_COL].map(y_unit)
    return df

_build_die_df(
    uid_train_die, xs_train_die[DIE_KEY_COL].values, xs_train_die['position'].values,
    oof_die_prob, oof_die_reg, oof_die_pred, y_train_unit,
).to_csv(os.path.join(OUT_DIR, 'oof_die.csv'), index=False)
_build_die_df(
    uid_val_die, xs_val_die[DIE_KEY_COL].values, xs_val_die['position'].values,
    val_die_prob, val_die_reg, val_die_pred, y_val_unit,
).to_csv(os.path.join(OUT_DIR, 'val_die.csv'), index=False)
_build_die_df(
    uid_test_die, xs_test_die[DIE_KEY_COL].values, xs_test_die['position'].values,
    test_die_prob, test_die_reg, test_die_pred, y_test_unit,
).to_csv(os.path.join(OUT_DIR, 'test_die.csv'), index=False)

def _build_unit_df(unit_pred_s, y_unit):
    return pd.DataFrame({
        KEY_COL:    unit_pred_s.index.values,
        'pred':     unit_pred_s.values,
        TARGET_COL: y_unit.reindex(unit_pred_s.index).values,
    })

_build_unit_df(oof_unit_s,  y_train_unit).to_csv(os.path.join(OUT_DIR, 'oof_unit.csv'),  index=False)
_build_unit_df(val_unit_s,  y_val_unit ).to_csv(os.path.join(OUT_DIR, 'val_unit.csv'),  index=False)
_build_unit_df(test_unit_s, y_test_unit).to_csv(os.path.join(OUT_DIR, 'test_unit.csv'), index=False)

with open(os.path.join(OUT_DIR, 'best_params.json'), 'w', encoding='utf-8') as f:
    json.dump({
        'best_value':   study.best_value,
        'best_params':  study.best_trial.params,
        'hp_best':      hp_best,
        'best_w0':      best_w0,
        'best_reg_obj': best_reg_obj,
        'best_clf_spw': best_clf_spw,
    }, f, indent=2, ensure_ascii=False, default=str)

meta = {
    'exp_id':            EXP_ID,
    'exp_memo':          EXP_MEMO,
    'model':             'Reverse Two-Stage (lgbm reg → lgbm clf, weighted MSE, log1p)',
    'path_type':         'B (reverse: reg → clf)',
    'target_transform':  TARGET_TRANSFORM,
    'die_to_unit_agg':   'mean',
    'training_level':    'die-level broadcast',
    'n_trials':          N_TRIALS,
    'n_folds':           N_FOLDS,
    'oof_rmse':          oof_rmse,
    'val_rmse':          val_rmse,
    'test_rmse':         test_rmse,
    'preprocess_PARAMS': PARAMS,
    'effective_pp_params': pp['effective_params'],
    'best_params':       study.best_trial.params,
    'CLIP_Y_EXTREME':    CLIP_Y_EXTREME,
    'feat_cols_clean_n': len(feat_cols_clean),
    'SEED':              int(SEED),
}
with open(os.path.join(OUT_DIR, 'meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print(f'저장 완료: {OUT_DIR}')
for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:35s}  {sz:>10,.1f} KB')

저장 완료: c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\two_stage_reverse
  best_params.json                            1.1 KB
  meta.json                                   1.9 KB
  oof_die.csv                             9,884.6 KB
  oof_unit.csv                              970.7 KB
  optuna_jh_ts-reverse-hpo-001.db           808.0 KB
  test_die.csv                            3,301.4 KB
  test_unit.csv                             323.5 KB
  val_die.csv                             3,301.1 KB
  val_unit.csv                              323.4 KB


## 10. 요약

In [10]:
print('=' * 75)
print(f' Reverse Two-Stage (path B) + Optuna HPO — 결과 요약')
print('=' * 75)
print(f'  EXP_ID            : {EXP_ID}')
print(f'  PP source         : 03b log1p preset')
print(f'  N_TRIALS / N_FOLDS: {N_TRIALS} / {N_FOLDS}')
print(f'  best HP search    : LGBM 11종 + w0 + reg_obj + clf_spw (14 axis)')
print(f'  best w0           : {best_w0:.4f}')
print(f'  best reg_obj      : {best_reg_obj}')
print(f'  best clf_spw      : {best_clf_spw}')
print(f'  feat cols (clean) : {len(feat_cols_clean)}')
print('-' * 75)
print(f'  {"":10s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"unit RMSE":10s}  {oof_rmse:11.6f}  {val_rmse:11.6f}  {test_rmse:11.6f}')
print('-' * 75)
print(f'  → 03b (path A, val=0.005718) 대비 plateau 안 들어오면 다양성 후보.')
print(f'  → 잔차 corr 검사로 stacking 추가 가치 정량 확인 권장.')
print('=' * 75)

 Reverse Two-Stage (path B) + Optuna HPO — 결과 요약
  EXP_ID            : ts-reverse-hpo-001
  PP source         : 03b log1p preset
  N_TRIALS / N_FOLDS: 300 / 5
  best HP search    : LGBM 11종 + w0 + reg_obj + clf_spw (14 axis)
  best w0           : 0.1767
  best reg_obj      : regression
  best clf_spw      : 2.43
  feat cols (clean) : 568
---------------------------------------------------------------------------
                      OOF          val         test
  unit RMSE      0.005495     0.005709     0.008412
---------------------------------------------------------------------------
  → 03b (path A, val=0.005718) 대비 plateau 안 들어오면 다양성 후보.
  → 잔차 corr 검사로 stacking 추가 가치 정량 확인 권장.
